# B3.13 Update

From April 14 2025 -- now (May 9 2025)

Check metadata for old sequences to see if anything has been added

In [ ]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = home + "Combinations/11-01-2021--06-05-2025_B3_13/"
downloads = home + "Andersen/"
temp_files = downloads + "temp/"
complete_files = downloads + "complete/"

update_date = "06-10-2025"

os.chdir(originals)

## Andersen Lab

In [ ]:
# Andersen

# Read metadata

os.chdir(home)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = downloads + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2021, 1, 1).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= datetime(2025, 6, 10).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)

genotypes = ["B3.13"]

571


### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

In [ ]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results = genoflu_results.rename(columns={"sample": "Run"})

metadata = metadata.merge(genoflu_results, on="Run", how="inner")

# Get only the genotypes we want

metadata = metadata[metadata["Genotype"] == genotypes[0]]

# metadata = metadata.merge(b313_only, on=["Run"], how="inner")

print(len(metadata)) 

# display(metadata)

           sample                 date       File Name Genotype  \
8     SRR32125554  2025-04-08_15-50-41  SRR32125554.fa     D1.1   
19    SRR32006903  2025-04-08_15-49-57  SRR32006903.fa     D1.1   
26    SRR31347623  2025-04-08_15-18-21  SRR31347623.fa     D1.1   
28    SRR32973801  2025-04-08_16-23-56  SRR32973801.fa     D1.1   
36    SRR32416328  2025-04-08_16-01-58  SRR32416328.fa     D1.1   
...           ...                  ...             ...      ...   
8952  SRR33370219  2025-05-02_06-24-03  SRR33370219.fa     D1.1   
8953  SRR33370220  2025-05-02_06-24-04  SRR33370220.fa     D1.1   
8954  SRR33370221  2025-05-02_06-24-03  SRR33370221.fa     D1.1   
8955  SRR33370222  2025-05-02_06-24-03  SRR33370222.fa     D1.1   
8956  SRR33370223  2025-05-02_06-24-03  SRR33370223.fa     D1.1   

                            Genotype List Used, >=98.0%  \
8     PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...   
19    PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...   
26    PB2:am24, PB

In [ ]:
# Get specific geolocation from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
# genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

print(genbank_mapping)

metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir("C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/")
state_ref = pd.read_csv("states_ref.csv")
metadata_genbank["Geo_Location"] = metadata_genbank["name_state"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x, 'Country'].iloc[0] + "-" + x if x in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x, 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x, 'Abbreviation'].iloc[0] if x in state_ref["State"].values else x)


print(genbank_mapping["name_state"])
print(len(metadata_genbank))
display(metadata_genbank) # No state information since 3/18/2025?

                     seg_file  \
0       SRR28752446_HA_cns.fa   
1       SRR28752446_MP_cns.fa   
2       SRR28752446_NA_cns.fa   
3       SRR28752446_NP_cns.fa   
4       SRR28752446_NS_cns.fa   
...                       ...   
40773   SRR33124777_NP_cns.fa   
40774   SRR33124777_NS_cns.fa   
40775   SRR33124777_PA_cns.fa   
40776  SRR33124777_PB1_cns.fa   
40777  SRR33124777_PB2_cns.fa   

                                            seg_seq_name      sra_run  seg  \
0      Consensus_SRR28752446_HA_cns_threshold_0.5_qua...  SRR28752446   HA   
1      Consensus_SRR28752446_MP_cns_threshold_0.5_qua...  SRR28752446   MP   
2      Consensus_SRR28752446_NA_cns_threshold_0.5_qua...  SRR28752446  NaN   
3      Consensus_SRR28752446_NP_cns_threshold_0.5_qua...  SRR28752446   NP   
4      Consensus_SRR28752446_NS_cns_threshold_0.5_qua...  SRR28752446   NS   
...                                                  ...          ...  ...   
40773  Consensus_SRR33124777_NP_cns_threshold_0.5_qua... 

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Genotype Mismatch List,Genotype Average Depth of Coverage List,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state
0,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,2025,...,"8, 10, 2, 9, 5, 9, 16, 7",Ran on FASTA - No Coverage Report,SRR33124722_HA_cns.fa,Consensus_SRR33124722_HA_cns_threshold_0.5_qua...,SRR33124722,HA,PV572785.1,4,A/cattle/NV/25-006542-003-original/2025,NV
1,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,2025,...,"8, 10, 2, 9, 5, 9, 16, 7",Ran on FASTA - No Coverage Report,SRR33124722_MP_cns.fa,Consensus_SRR33124722_MP_cns_threshold_0.5_qua...,SRR33124722,MP,PV572788.1,7,A/cattle/NV/25-006542-003-original/2025,NV
2,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,2025,...,"8, 10, 2, 9, 5, 9, 16, 7",Ran on FASTA - No Coverage Report,SRR33124722_NA_cns.fa,Consensus_SRR33124722_NA_cns_threshold_0.5_qua...,SRR33124722,NaN,PV572787.1,6,A/cattle/NV/25-006542-003-original/2025,NV
3,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,2025,...,"8, 10, 2, 9, 5, 9, 16, 7",Ran on FASTA - No Coverage Report,SRR33124722_NP_cns.fa,Consensus_SRR33124722_NP_cns_threshold_0.5_qua...,SRR33124722,NP,PV572786.1,5,A/cattle/NV/25-006542-003-original/2025,NV
4,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,2025,...,"8, 10, 2, 9, 5, 9, 16, 7",Ran on FASTA - No Coverage Report,SRR33124722_NS_cns.fa,Consensus_SRR33124722_NS_cns_threshold_0.5_qua...,SRR33124722,NS,PV572789.1,8,A/cattle/NV/25-006542-003-original/2025,NV
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025,...,"9, 1, 7, 4, 7, 18, 9, 12",Ran on FASTA - No Coverage Report,SRR33124777_NP_cns.fa,Consensus_SRR33124777_NP_cns_threshold_0.5_qua...,SRR33124777,NP,PV572754.1,5,A/cattle/NV/25-006535-001-original/2025,NV
260,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025,...,"9, 1, 7, 4, 7, 18, 9, 12",Ran on FASTA - No Coverage Report,SRR33124777_NS_cns.fa,Consensus_SRR33124777_NS_cns_threshold_0.5_qua...,SRR33124777,NS,PV572757.1,8,A/cattle/NV/25-006535-001-original/2025,NV
261,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025,...,"9, 1, 7, 4, 7, 18, 9, 12",Ran on FASTA - No Coverage Report,SRR33124777_PA_cns.fa,Consensus_SRR33124777_PA_cns_threshold_0.5_qua...,SRR33124777,PA,PV572752.1,3,A/cattle/NV/25-006535-001-original/2025,NV
262,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,2025,...,"9, 1, 7, 4, 7, 18, 9, 12",Ran on FASTA - No Coverage Report,SRR33124777_PB1_cns.fa,Consensus_SRR33124777_PB1_cns_threshold_0.5_qu...,SRR33124777,PB1,PV572751.1,2,A/cattle/NV/25-006535-001-original/2025,NV


In [ ]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"


In [ ]:
# # Get all dates
# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# # Save this so we don't have to do it again

# os.chdir(temp_files)
# metadata_genbank.to_csv("metadata_genbank_" + update_date + ".csv")

In [ ]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(downloads + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank_5-14-2025.csv")
# os.chdir(temp_files)

In [9]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['cattle', 'goose', 'chukar', 'guineafowl', 'chicken', 'duck']
[]
                 avian               cattle        feline   other_mammal  \
0     great_horned_owl            dairy_cow           cat     deer mouse   
1         common_raven               cattle  domestic_cat    house_mouse   
2        cooper's_hawk  cattle milk product     feral_cat          skunk   
3         coopers_hawk          bovine_milk        feline  striped_skunk   
4              peafowl              bovine   domestic-cat     norway rat   
..                 ...                  ...           ...            ...   
403   blue-winged teal                  NaN           NaN            NaN   
404       lesser scaup                  NaN           NaN            NaN   
405   bonaparte's gull                  NaN           NaN            NaN   
406  rough-legged hawk                  NaN           NaN            NaN   
407         sand crane                  NaN           NaN            NaN   

          human      

In [10]:
# Get animals from animal reference
os.chdir(home)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type

metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [ ]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date"] = collection_date
        else:
            parsed_date = collection_date.split("|")[-1]
            parsed_date = dateutil.parser.parse(parsed_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Geo_Location"] + "|" + metadata_genbank["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

os.chdir(temp_files)

metadata_genbank.to_csv("metadata_genbank_D1_1_" + update_date + ".csv")

display(metadata_genbank)

,Unnamed: 0,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,...,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Collection_Date_Specific,Host_Type,years,Name
0,0,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,...,SRR33124722,HA,PV572785.1,4,A/cattle/NV/25-006542-003-original/2025,NV,2025-02-18,cattle,2025,>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...
1,1,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,...,SRR33124722,MP,PV572788.1,7,A/cattle/NV/25-006542-003-original/2025,NV,2025-02-18,cattle,2025,>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...
2,2,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,...,SRR33124722,NaN,PV572787.1,6,A/cattle/NV/25-006542-003-original/2025,NV,2025-02-18,cattle,2025,>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...
3,3,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,...,SRR33124722,NP,PV572786.1,5,A/cattle/NV/25-006542-003-original/2025,NV,2025-02-18,cattle,2025,>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...
4,4,SRR33124722,WGS,248.04,158717244,PRJNA980729,SAMN47941357,Viral,55355880,USDA-NVSL,...,SRR33124722,NS,PV572789.1,8,A/cattle/NV/25-006542-003-original/2025,NV,2025-02-18,cattle,2025,>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,259,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,...,SRR33124777,NP,PV572754.1,5,A/cattle/NV/25-006535-001-original/2025,NV,2025-02-17,cattle,2025,>A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...
260,260,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,...,SRR33124777,NS,PV572757.1,8,A/cattle/NV/25-006535-001-original/2025,NV,2025-02-17,cattle,2025,>A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...
261,261,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,...,SRR33124777,PA,PV572752.1,3,A/cattle/NV/25-006535-001-original/2025,NV,2025-02-17,cattle,2025,>A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...
262,262,SRR33124777,WGS,242.97,282434514,PRJNA980729,SAMN47941352,Viral,97835669,USDA-NVSL,...,SRR33124777,PB1,PV572751.1,2,A/cattle/NV/25-006535-001-original/2025,NV,2025-02-17,cattle,2025,>A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...


In [14]:
# Get information to create the fasta files

fasta_folder = downloads + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype_y"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [ ]:
# Create fasta files 

# os.chdir(downloads + "complete/B3_13_D1_1/" + "04-14-2025--05-14-2025_D1_1/")
os.chdir(originals)

for pair in fasta_files.keys():
    output_path = originals + "complete/B3_13_D1_1/" + "04-14-2025--05-14-2025_D1_1/" + pair + "_andersen_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-18|cattle|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GUINEAFOWL/PA/25-006346-011/2025|H5N1|2025-02-20|avian|D1.1
>A/GOOSE/OH/25-0

## GISAID

In [17]:
# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
complete_files = home + "GISAID/complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1_northa/"

update_date = "05-14-2025"

os.chdir(originals)

In [18]:
# Have user type in username and password

# username = input("Username: ")
# password = input("Password: ")
# browser = input("Browser: ")
# sleep_time = input("Seconds to sleep in between clicks: ")
start_date = input("Start date (format: YYYY-MM-DD): ")
end_date = input("End date (format: YYYY-MM-DD): ")

genotypes = ["D1.1"]

In [19]:
# open_gisaid(username, password, browser, sleep_time, start_date, end_date)

In [20]:
all_metadata_files = []
all_fasta_files = []

# Grab files
for dirpath, dirs, files in os.walk(downloads):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)

unique_animals_all = [] # find unique animals to sort them later

os.chdir(home)

# Separate fastas by segment
segment_fastas = []
unique_segments = []
for i, fasta in enumerate(all_fasta_files):

    metadata = all_metadata_files[i]

    # print(fasta.loc[i, "Isolate_Name"])
    
    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_seg(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment

    segment_fastas.append(fastas)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)


c:\Users\maksiaevai.NCBI_NT\Documents\Avian_Flu\utils.py:471: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
c:\Users\maksiaevai.NCBI_NT\Documents\Avian_Flu\utils.py:475: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
c:\Users\maksiaevai.NCBI_NT\Documents\Avian_Flu\utils.py:471: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in

In [21]:
huge_fasta = pd.DataFrame()

for fastas in segment_fastas: 
    # print(len(fastas))
    # break
    for f in fastas: 
        # print(f)
        # break 
        huge_fasta = pd.concat([huge_fasta, f])

# print(huge_fasta.columns)

# Now separate huge_fasta into 16 fastas
big_fastas = []

# print(huge_fasta)

# genotypes = ["B3.13", "D1.1"] # , "D1.3"]
for gen in genotypes:
    # print(gen)
    big_fasta = huge_fasta[huge_fasta["Genotype"] == gen]
    # print(gen)
    for seg in unique_segments:
        seg_specific_fasta = big_fasta[big_fasta["Segment"] == seg]
        big_fastas.append(seg_specific_fasta)


# Now that we have 16 fastas, write the files
for fasta in big_fastas:

    # Create a dictionary to create a file
    fasta_df = fasta[["New_Name", "Sequence"]]
    fasta_dict = pd.Series(fasta_df.Sequence.values,index=fasta_df.New_Name).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = complete_files + fasta["Genotype"].values[0] + "_" + fasta["Segment"].values[0] + "_GISAID_" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            output_file.write(item)
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

Succeeded in finding results for genotype:  D1.1
Succeeded in finding results for genotype:  D1.1
Succeeded in finding results for genotype:  D1.1
Succeeded in finding results for genotype:  D1.1
Succeeded in finding results for genotype:  D1.1
Succeeded in finding results for genotype:  D1.1
Succeeded in finding results for genotype:  D1.1
Succeeded in finding results for genotype:  D1.1


In [22]:
# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(home)
animals_df.to_csv("animals_ref_to_sort.csv")

['greater_scaup', 'turkey_vulture', 'herring_gull', 'nasua_nasua', 'mute_swan', 'south_georgia_shag', 'bald_eagle', 'procellaria_aequinoctialis', 'pluvialis_dominica', 'burmeisters_porpoise', 'american_crow', 'owl', 'avian', 'great_grabe', 'swan', 'mallard', 'band-tailed_gull', 'falcon', 'tundra_swan', 'antarctic_fur_seal', 'bufflehead', 'sula_leucogaster', 'pelican', 'sea_lion', 'cackling_goose', 'western_sandpiper', 'gull', 'south_american_sea_lion', 'backyard_duck', 'guanay_cormorant', 'crow', 'red_fox', 'northern_pintail', 'american_black_duck', 'bobcat', 'sandhill_crane', 'flamingo', 'royal_tern', 'numida_meleagris', 'dairy_cow', 'pelecanus', 'cat', 'common_loon', "cooper's_hawk", 'western_gull', 'duck', 'great_grebe', 'kelp_gull', 'rough-legged_hawk', 'sand_crane', 'peregrine_falcon', 'chiloe_wigeon', 'south_polar_skua', "ross's_goose", 'sanderling', "franklin's_gull", 'brown_booby', 'wood_duck', 'goose', 'otaria_flavescens', 'american_green-winged_teal', 'guineafowl', 'south_ame

## De-Duplication

In [28]:
# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
combined_files = home + "Combinations/GISAID_Andersen/B3_13_D1_1/04-14-2025--05-14-2025_D1_1/"
downloads = home + "Andersen/"
temp_files = home + "Andersen/temp/"
complete_files = home + "Andersen/complete/"

update_date = "05-14-2025"

gisaid = home + "GISAID/complete/B3_13_D1_1/04-14-2025--05-14-2025_D1_1_northa/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [33]:
dfs_andersen = create_dataframes(complete_files + "B3_13_D1_1/04-14-2025--05-14-2025_D1_1/")

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [34]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
defaultdict(<class 'list'>, {'D1.1_HA': [    isolate_partial                                        full_header  \
0        006542-003  >A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...   
1        006542-003  >A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...   
2        006542-003  >A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...   
3        006542-003  >A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...   
4        006542-003  >A/CATTLE/NV/25-006542-003/2025|H5N1|2025-02-1...   
..              ...                                                ...   
259      006535-001  >A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...   
260      006535-001  >A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...   
261      006535-001  >A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...   
262      006535-001  >A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...   
263      006535-001  >A/CATTLE/NV/25-006535-001/2025|H5N1|2025-02-1...   

    

In [35]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                gisaid_df = dfs_gisaid[gisaid_key][0]
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                full_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")
                full_dfs[andersen_key].append(full_df)

print(full_dfs)

defaultdict(<class 'list'>, {'D1.1_HA': [    isolate_partial                                        full_header  \
55       007078-002  >A/DUCK/OH/25-007078-002/2025|H5N1|2025-02-24|...   
63       007078-001  >A/DUCK/OH/25-007078-001/2025|H5N1|2025-02-24|...   
79       006674-002  >A/DUCK/ME/25-006674-002/2025|H5N1|2025-02-24|...   
135      007056-001  >A/CHICKEN/NC/25-007056-001/2025|H5N1|2025-02-...   
231      006346-008  >A/CHICKEN/PA/25-006346-008/2025|H5N1|2025-02-...   
..              ...                                                ...   
640      005056-001  >A/canada_goose/USA/005056-001/2025|H5N1|2025|...   
641      006228-001  >A/snow_goose/USA/006228-001/2025|H5N1|2025|av...   
642      004961-001  >A/canada_goose/USA/004961-001/2025|H5N1|2025|...   
643      005009-001  >A/canada_goose/USA/005009-001/2025|H5N1|2025|...   
644      005011-001  >A/wood_duck/USA/005011-001/2025|H5N1|2025|avi...   

                                              sequence  
55   ATGGAAAA

In [36]:
# Create FASTA files per segment

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + "_D1_1.fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
